# Profit prediction

In [48]:
import pandas as pd
import matplotlib.pyplot as plt

In [70]:
df = pd.read_csv("profit_prediction_dataset.csv")
df.head()

,Marketing Spend,Administration,Transport,Area,Profit
0,114523.61,136897.80,471784.10,Dhaka,192261.83
1,162597.70,151377.59,443898.53,Ctg,191792.06
2,153441.51,101145.55,407934.54,Rangpur,191050.39
3,144372.41,118671.85,383199.62,Dhaka,182901.99
4,142107.34,91391.77,366168.42,Rangpur,166187.94


# Null value handling

In [71]:
df.isnull().sum()

Marketing Spend    0
Administration     0
Transport          1
Area               1
Profit             0
dtype: int64

In [72]:
df.dtypes

Marketing Spend    float64
Administration     float64
Transport          float64
Area                object
Profit             float64
dtype: object

In [73]:
transport_mean = df.Transport.mean()
transport_mean

np.float64(210096.7748979592)

In [74]:
df.Transport = df.Transport.fillna(transport_mean)
df.isnull().sum()

Marketing Spend    0
Administration     0
Transport          0
Area               1
Profit             0
dtype: int64

In [80]:
df['Area'].fillna(df["Area"].mode()[0], inplace=True)
df.isnull().sum()

Marketing Spend    0
Administration     0
Transport          0
Area               0
Profit             0
dtype: int64

# Separate x, y

In [84]:
x = df.drop(["Profit"], axis=1)
y = df['Profit']

# Train test split

In [92]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=12)
xtrain.shape  # showing 37 rows out of 49

(37, 4)

In [93]:
xtest.shape

(13, 4)

# One hot encoding

In [106]:
area = pd.get_dummies(xtrain['Area'], dtype=int)
area.head()

,Ctg,Dhaka,Rangpur
15,0,1,0
31,0,1,0
14,0,0,1
26,0,0,1
9,1,0,0


In [107]:
xtrain = pd.concat([xtrain, area], axis=1)
xtrain.head()

,Marketing Spend,Administration,Transport,Area,Ctg,Dhaka,Rangpur
15,165349.20,122616.84,261776.230000,Dhaka,0,1,0
31,61136.38,152701.92,88218.230000,Dhaka,0,1,0
14,119943.24,156547.42,210096.774898,Rangpur,0,0,1
26,75328.87,144135.98,134050.070000,Rangpur,0,0,1
9,123334.88,108679.17,304981.620000,Ctg,1,0,0


In [108]:
xtrain = xtrain.drop(['Area'], axis=1)
xtrain.head()

,Marketing Spend,Administration,Transport,Ctg,Dhaka,Rangpur
15,165349.20,122616.84,261776.230000,0,1,0
31,61136.38,152701.92,88218.230000,0,1,0
14,119943.24,156547.42,210096.774898,0,0,1
26,75328.87,144135.98,134050.070000,0,0,1
9,123334.88,108679.17,304981.620000,1,0,0


In [109]:
test_area = pd.get_dummies(xtest['Area'], dtype=int)
xtest = pd.concat([xtest, test_area], axis=1)
xtest = xtest.drop(['Area'], axis=1)
xtest.head()

,Marketing Spend,Administration,Transport,Ctg,Dhaka,Rangpur
28,66051.52,182645.56,118148.20,0,0,1
46,1315.46,115816.21,297114.46,0,0,1
7,130298.13,145530.06,323876.68,0,0,1
41,27892.92,84710.77,164470.71,0,0,1
36,28663.76,127056.21,201126.82,0,0,1


# Linear Regression

In [111]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

In [112]:
lr.fit(xtrain, ytrain)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [113]:
y_pred = lr.predict(xtest)
y_pred

array([104036.02093113,  75862.31334978, 161179.69044521,  76129.82057325,
        84161.70249461, 100881.2281024 , 128270.80908453,  40541.15609299,
       103028.04632814, 155122.27756814, 135827.56561563,  72696.88514176,
        85611.73813457])

In [114]:
ytest

28    103282.38
46     49490.75
7     155752.60
41     77798.83
36     90708.19
29    101004.64
21    111313.02
48     35673.41
19    122776.86
8     152211.77
17    125370.37
38     81229.06
37     89949.14
Name: Profit, dtype: float64

In [115]:
from sklearn.metrics import r2_score
r2_score(ytest, y_pred)

0.8868005975081162